# XTTS v2 TTS 엔진 테스트

ReadMate 파이프라인에서 XTTS v2 기반 음성 합성을 테스트합니다.

## Cell 1: 환경 설정

In [ ]:
import sys

sys.path.insert(0, '/c/Users/user/workspaces/read-mate')

import logging

logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(name)s - %(levelname)s - %(message)s')

print('환경 설정 완료')

## Cell 2: XTTSEngine 초기화

첫 실행 시 모델을 HuggingFace Hub에서 다운로드합니다 (~1.8GB, 5~10분 소요).

In [ ]:
from src.services.tts_service import XTTSEngine

print('XTTSEngine 초기화 중...')
engine = XTTSEngine()
print('✓ XTTSEngine 초기화 완료')

## Cell 3: 프리셋 목록 확인

In [ ]:
presets = engine.list_presets()
print(f'사용 가능한 프리셋: {presets}')

if not presets:
    print('⚠️  프리셋이 없습니다. assets/voices/default.wav를 추가하세요.')
else:
    print(f'✓ 총 {len(presets)}개 프리셋 로드됨')

## Cell 4: 한국어 텍스트 합성 (프리셋 없이)

프리셋 파일이 없어도 XTTS v2는 기본 화자로 합성합니다.

In [ ]:
test_text = "안녕하세요. ReadMate입니다. 오늘의 문서를 읽어드리겠습니다."

print(f'합성할 텍스트: {test_text}')
print('합성 중...')

try:
    result = engine.synthesize(test_text, voice_preset='default')
    print('✓ 합성 완료')
    print(f'  오디오 경로: {result.audio_path}')
    print(f'  재생 시간: {result.duration_sec:.2f}초')
    print(f'  엔진: {result.engine}')
    print(f'  프리셋: {result.voice_preset}')
except RuntimeError as e:
    print(f'✗ 합성 실패: {e}')

## Cell 5: 오디오 재생

In [ ]:
from IPython.display import Audio

if 'result' in locals():
    Audio(result.audio_path)

## Cell 6: 긴 텍스트 테스트 (split_sentences 동작 확인)

In [ ]:
long_text = """
인공지능 기술의 발전으로 자연어 처리 분야에서 눈부신 성과가 이루어지고 있습니다.
특히 대형 언어 모델의 등장으로 텍스트 이해와 생성 능력이 비약적으로 향상되었습니다.
이러한 기술은 교육, 의료, 법률 등 다양한 분야에서 활용되고 있습니다.
음성 합성 기술도 마찬가지로 빠르게 발전하고 있으며, 더 자연스럽고 표현력 있는 음성을 만들어낼 수 있게 되었습니다.
"""

print(f'긴 텍스트 길이: {len(long_text)}자')
print('합성 중... (긴 텍스트이므로 시간이 걸릴 수 있습니다)')

try:
    result_long = engine.synthesize(long_text, voice_preset='default')
    print('✓ 합성 완료')
    print(f'  재생 시간: {result_long.duration_sec:.2f}초')
    print(f'  오디오 경로: {result_long.audio_path}')
except RuntimeError as e:
    print(f'✗ 합성 실패: {e}')

## Cell 7: 긴 텍스트 오디오 재생

In [ ]:
if 'result_long' in locals():
    Audio(result_long.audio_path)

## Cell 8: 생성된 오디오 파일 확인

In [ ]:
from pathlib import Path

output_dir = Path('/c/Users/user/workspaces/read-mate/outputs/tts')
if output_dir.exists():
    wav_files = list(output_dir.glob('*.wav'))
    print(f'생성된 WAV 파일: {len(wav_files)}개')
    for f in sorted(wav_files, key=lambda x: x.stat().st_mtime, reverse=True)[:5]:
        size_mb = f.stat().st_size / (1024 * 1024)
        print(f'  - {f.name} ({size_mb:.2f}MB)')
else:
    print('⚠️  outputs/tts 디렉토리가 없습니다.')